# [방법론 1 보강 · 2026-10-26] 정책 경로 내보내기 — β 4개 × 검증·테스트 × 원본·워밍업

`crypto_attribution_rt1006`와 **같은 준비 셀·같은 `policy_path` 함수**로 각 β 정책의 스텝별 보유 비중 경로를 다시 만들어,
맥에서 분석할 수 있도록 압축 파일로 저장한다. 학습·평가 설정은 하나도 바꾸지 않는다(결과를 보고 고르는 단계가 없음).

이 파일로 맥에서 하는 일:
1. 같은 노출 BTC·등가중 비교와 MDD를 커밋된 스크립트로 재현(감사에서 '재현 불가'였던 수치)
2. 분해 층별 블록 bootstrap 신뢰구간, 무작위(위약) 정책 귀무분포, 분해 순서 민감도 — 사전 등록 후

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 30~60분. 학습 없음.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/policy_paths/` (npz 18개 + `meta.json`, 합계 수십 MB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·평가 함수 (β=30으로 치환) — 원본 crypto_beta_robustness_colab(커밋 0e86a41) 셀 재사용
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


In [ ]:
import sys, subprocess
REPO = Path('/content/ToBeRich')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/winbean-kwon/ToBeRich.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '4f3f508df45a1ea7287cb6cec0527f96a891e323'], check=True)
sys.path.insert(0, str(REPO))
from crypto.experiments.attribution import attribute, portfolio_log_growth
print('분해 모듈 로드 완료')


In [ ]:
%%time
WARMUP = 14 * 48
BETAS = [-90, -30, 30, 90]
POOL_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
models = {b: PPO.load(str(POOL_V4 / f'agent_beta_{b}.zip')) for b in BETAS}


def policy_path(model, buckets, start):
    """start_state_audit의 run_policy와 같은 루프 — 스텝별 보유 비중·수익률·거래 가능 마스크와 시작 비중을 기록."""
    if start == 'warmup':
        i0 = all_buckets.index(buckets[0]); lo = max(0, i0 - WARMUP)
        run_bk, skip = all_buckets[lo:i0] + list(buckets), i0 - lo
    else:
        run_bk, skip = list(buckets), 0
    env = SparseGatedCryptoPortfolioEnv(buckets=run_bk, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset()
    if start == 'warmup':
        env.weights = np.zeros(env.n_assets, dtype=np.float32); obs = env._get_obs()
    init = env.weights.copy()
    Ws, Rs, Ts, done = [], [], [], False
    while not done:
        b = env.buckets[env.t]
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, top_idx = env.action_to_weights(a)
        w = (1 - ALPHA) * env.weights + ALPHA * tw
        w_held = _mask_untradable(w, env._tradable(b), renorm=False)
        if env.t == skip:
            init = env.weights.copy()   # 평가 시작 시점의 보유 비중
        Ws.append(w_held); Rs.append(env._asset_ret(b)); Ts.append(env._tradable(b))
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    v = np.asarray(env.value_history[skip:])
    return np.asarray(Ws[skip:], np.float64), np.asarray(Rs[skip:], np.float64), np.asarray(Ts[skip:]), init, v[-1] / v[0] - 1


In [ ]:
%%time
import json as _json
OUT = CRYPTO_DIR / 'policy_paths'
OUT.mkdir(parents=True, exist_ok=True)
summary = []
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    R_ref = T_ref = None
    for beta, model in models.items():
        for start in ('orig', 'warmup'):
            W, R, T, init, ret = policy_path(model, bk, start)
            g, _ = portfolio_log_growth(W, R, TC, init)
            assert abs(np.expm1(g) - ret) < 1e-6, (split, beta, start, np.expm1(g), ret)
            if R_ref is None:
                R_ref, T_ref = R, T
            assert np.allclose(R, R_ref) and np.array_equal(T, T_ref)
            np.savez_compressed(OUT / f'{split}_beta{beta}_{start}.npz', W=W, init=init, ret=np.float64(ret))
            summary.append({'구간': split, 'β': beta, '시작': start, '수익': ret, '평균 노출': float(W.sum(1).mean())})
            print(f'{split} β={beta} {start}: {ret:+.2%} (평균 노출 {W.sum(1).mean():.3f}) 저장')
    np.savez_compressed(OUT / f'{split}_market.npz', R=R_ref, T=T_ref,
                        ts=np.array([str(bucket_to_ts(b)) for b in bk]))
meta = {'symbols': list(symbols), 'TC': float(TC), 'ALPHA': float(ALPHA), 'TOP_K': int(TOP_K), 'WARMUP_STEPS': int(WARMUP),
        'BETAS': BETAS, 'valid': [str(bucket_to_ts(valid_buckets[0])), str(bucket_to_ts(valid_buckets[-1]))],
        'test': [str(bucket_to_ts(test_buckets[0])), str(bucket_to_ts(test_buckets[-1]))]}
(OUT / 'meta.json').write_text(_json.dumps(meta, ensure_ascii=False, indent=1))
pd.DataFrame(summary).to_csv(OUT / 'summary.csv', index=False, encoding='utf-8-sig')
print(pd.DataFrame(summary).to_string())
print('\n저장 완료:', OUT)
